# OTEL Synthetic Data Generator

Generate Hive-partitioned span parquet for **Panel-Viz** and **Dask_S3_Validation**.

## Shared layout (must match validation)

```text
s3://$S3_BUCKET/$PREFIX/spans/date=YYYY-MM-DD/batch_*.parquet
```

| | |
|--|--|
| Prefix | `otel-notebook` (same default as `cluster_env` / validation) |
| Endpoint/creds | JupyterHub env via `cluster_env` (local RustFS/MinIO) |
| Marker | `s3://$S3_BUCKET/_active_dataset.json` with `"dataset": "<PREFIX>"` |

## Lab scale (single-node defaults)

| Knob | Default | Why |
|------|---------|-----|
| `TOTAL_SPANS` | **100_000** | Enough for Dask; fits tinybox-class workers |
| `CHUNK_SIZE` | **50_000** | Few write batches |
| `DURATION_DAYS` | **1** | One `date=` partition → fast LIST for validation |

Stress: `TOTAL_SPANS=2000000 DURATION_DAYS=7`.

**Order:** run this → then **Dask_S3_Validation** (`SPANS_DATE` = printed date).

`cp /root/sample-notebooks/OTEL_Data_Generator.ipynb /root/`


## 0. Config from cluster env (shared with validation)


In [ ]:
import os, sys, json, time
from datetime import datetime, timedelta, timezone

import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import pyarrow.fs as pafs

for _p in ("/root/sample-notebooks", "/app", "/root"):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from cluster_env import load_cluster_config

CFG = load_cluster_config()
print(CFG.summary())

# Lab single-node defaults (override with env for stress)
TOTAL_SPANS = int(os.getenv("TOTAL_SPANS", os.getenv("OTEL_TOTAL_SPANS", "100000")))
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "50000"))
DURATION_DAYS = int(os.getenv("DURATION_DAYS", "1"))
COMPRESSION = os.getenv("PARQUET_COMPRESSION", "ZSTD")
COMPRESSION_LEVEL = int(os.getenv("PARQUET_COMPRESSION_LEVEL", "3"))
ROW_GROUP_SIZE = int(os.getenv("ROW_GROUP_SIZE", "50000"))

BUCKET = CFG.s3_bucket
PREFIX = (
    (CFG.otel_prefix or "").strip("/")
    or os.getenv("OTEL_PREFIX", "").strip("/")
    or os.getenv("PREFIX", "").strip("/")
    or "otel-notebook"
)
S3_ENDPOINT = CFG.s3_endpoint
S3_REGION = CFG.s3_region
SPANS_ROOT = f"{BUCKET}/{PREFIX}/spans"

print(f"\n=== paths (same as Dask_S3_Validation) ===")
print(f"  spans:   s3://{SPANS_ROOT}/date=YYYY-MM-DD/batch_*.parquet")
print(f"  marker:  s3://{BUCKET}/_active_dataset.json  dataset={PREFIX!r}")
print(f"  scale:   {TOTAL_SPANS:,} spans, chunk={CHUNK_SIZE:,}, days={DURATION_DAYS}")


## 1. Schema + vectorized batch generator


In [ ]:
import json
import random
import sys
import time
from datetime import datetime, timedelta, timezone

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pyarrow.fs as pafs

# Schema (30 columns — identical to production 1TB dataset)
_span_kind_dict = pa.dictionary(pa.int8(), pa.string())
_status_code_dict = pa.dictionary(pa.int8(), pa.string())

SPANS_SCHEMA = pa.schema([
    pa.field("trace_id", pa.string(), nullable=False),
    pa.field("span_id", pa.string(), nullable=False),
    pa.field("parent_span_id", pa.string(), nullable=True),
    pa.field("start_time_unix_nano", pa.int64(), nullable=False),
    pa.field("end_time_unix_nano", pa.int64(), nullable=False),
    pa.field("duration_ns", pa.int64(), nullable=False),
    pa.field("name", pa.string(), nullable=False),
    pa.field("kind", _span_kind_dict, nullable=False),
    pa.field("status_code", _status_code_dict, nullable=False),
    pa.field("status_message", pa.string(), nullable=True),
    pa.field("service_name", pa.string(), nullable=False),
    pa.field("service_namespace", pa.string(), nullable=True),
    pa.field("service_version", pa.string(), nullable=True),
    pa.field("host_name", pa.string(), nullable=True),
    pa.field("host_ip", pa.string(), nullable=True),
    pa.field("attributes_json", pa.string(), nullable=True),
    pa.field("resource_attributes_json", pa.string(), nullable=True),
    pa.field("http_method", pa.string(), nullable=True),
    pa.field("http_status_code", pa.int16(), nullable=True),
    pa.field("http_url", pa.string(), nullable=True),
    pa.field("http_route", pa.string(), nullable=True),
    pa.field("http_target", pa.string(), nullable=True),
    pa.field("db_system", pa.string(), nullable=True),
    pa.field("db_name", pa.string(), nullable=True),
    pa.field("db_operation", pa.string(), nullable=True),
    pa.field("db_statement", pa.string(), nullable=True),
    pa.field("rpc_system", pa.string(), nullable=True),
    pa.field("rpc_service", pa.string(), nullable=True),
    pa.field("rpc_method", pa.string(), nullable=True),
    pa.field("messaging_system", pa.string(), nullable=True),
    pa.field("messaging_destination", pa.string(), nullable=True),
    pa.field("messaging_operation", pa.string(), nullable=True),
    pa.field("exception_type", pa.string(), nullable=True),
    pa.field("exception_message", pa.string(), nullable=True),
    pa.field("events_count", pa.int32(), nullable=True),
    pa.field("links_count", pa.int32(), nullable=True),
])

# 12 service templates (identical to production)
SERVICE_TEMPLATES = [
    {"name": "api-gateway",          "kind": "SERVER",   "operations": ["HTTP request", "route", "rate-limit"]},
    {"name": "auth-service",         "kind": "SERVER",   "operations": ["validate_token", "refresh_token", "login", "logout"]},
    {"name": "user-service",         "kind": "SERVER",   "operations": ["get_user", "update_user", "list_users", "delete_user"]},
    {"name": "order-service",        "kind": "SERVER",   "operations": ["create_order", "get_order", "process_order", "cancel_order"]},
    {"name": "inventory-service",    "kind": "SERVER",   "operations": ["check_stock", "reserve_item", "release_item"]},
    {"name": "payment-service",      "kind": "SERVER",   "operations": ["process_payment", "refund", "validate_card"]},
    {"name": "notification-service", "kind": "PRODUCER", "operations": ["send_email", "send_sms", "send_push"]},
    {"name": "shipping-service",     "kind": "SERVER",   "operations": ["calculate_shipping", "create_shipment", "track"]},
    {"name": "analytics-service",    "kind": "CONSUMER", "operations": ["process_event", "aggregate", "report"]},
    {"name": "database",             "kind": "CLIENT",   "operations": ["SELECT", "INSERT", "UPDATE", "DELETE"]},
    {"name": "cache",                "kind": "CLIENT",   "operations": ["GET", "SET", "DEL", "MGET", "SCAN"]},
    {"name": "message-queue",        "kind": "PRODUCER", "operations": ["publish", "consume", "ack", "nack"]},
]

# Pre-compute flat arrays for vectorized lookups
SVC_NAMES = [s["name"] for s in SERVICE_TEMPLATES]
SVC_KINDS = [s["kind"] for s in SERVICE_TEMPLATES]
SVC_OPS, SVC_OP_IDX = [], []
for si, s in enumerate(SERVICE_TEMPLATES):
    for op in s["operations"]:
        SVC_OPS.append(op)
        SVC_OP_IDX.append(si)
SVC_OPS = np.array(SVC_OPS)
SVC_OP_IDX = np.array(SVC_OP_IDX)

HTTP_METHODS = np.array(["GET", "POST", "PUT", "DELETE", "PATCH"])
HTTP_ROUTES = np.array([
    "/api/users", "/api/users/{id}", "/api/orders", "/api/orders/{id}",
    "/api/products", "/api/products/{id}", "/api/auth/login", "/api/auth/logout",
    "/api/health", "/api/metrics", "/api/cart", "/api/checkout",
])
HTTP_STATUS_CODES = np.array([200, 200, 200, 200, 201, 204, 400, 401, 403, 404, 500, 502, 503], dtype=np.int16)

service_count = len(SERVICE_TEMPLATES)
print(f"Schema: {len(SPANS_SCHEMA)} columns")
print(f"Services: {service_count} templates, {len(SVC_OPS)} total operations")


In [ ]:
def generate_batch_vectorized(n, rng, start_ns, time_range_ns):
    """Generate n spans using vectorized numpy operations.

    Same methodology as the production 1TB K8s Indexed Job:
    - Exponential duration distribution (median ~50ms)
    - 5% error rate
    - Rejection sampling for service-operation matching
    - Full 30-column OTEL schema
    """
    # Timing: uniform start offset, exponential duration
    offsets_ns = rng.integers(0, time_range_ns, size=n, dtype=np.int64)
    span_starts = start_ns + offsets_ns
    durations = np.clip(
        rng.exponential(50_000_000, size=n).astype(np.int64),  # median ~50ms
        1_000_000, 30_000_000_000,  # 1ms to 30s
    )
    span_ends = span_starts + durations

    # Service assignment
    svc_indices = rng.integers(0, service_count, size=n)
    is_error = rng.random(size=n) < 0.05

    # Vectorized rejection sampling for service-operation matching
    op_global_indices = rng.integers(0, len(SVC_OPS), size=n)
    mismatched = SVC_OP_IDX[op_global_indices] != svc_indices
    retries = 0
    while mismatched.any() and retries < 20:
        new_indices = rng.integers(0, len(SVC_OPS), size=mismatched.sum())
        op_global_indices[mismatched] = new_indices
        mismatched = SVC_OP_IDX[op_global_indices] != svc_indices
        retries += 1
    if mismatched.any():
        for idx in np.where(mismatched)[0]:
            si = svc_indices[idx]
            candidates = np.where(SVC_OP_IDX == si)[0]
            op_global_indices[idx] = rng.choice(candidates)

    names = SVC_OPS[op_global_indices]
    kinds_arr = np.array(SVC_KINDS)[svc_indices]
    svc_names_arr = np.array(SVC_NAMES)[svc_indices]

    # Boolean masks for conditional fields
    is_server = kinds_arr == "SERVER"
    is_client = kinds_arr == "CLIENT"
    is_producer = kinds_arr == "PRODUCER"
    is_consumer = kinds_arr == "CONSUMER"
    is_msg = is_producer | is_consumer

    # HTTP fields (SERVER spans only)
    http_method_idx = rng.integers(0, len(HTTP_METHODS), size=n)
    http_route_idx = rng.integers(0, len(HTTP_ROUTES), size=n)
    http_status_idx = rng.integers(0, len(HTTP_STATUS_CODES), size=n)

    # Host fields
    host_nums = rng.integers(1, 11, size=n)
    ip_b = rng.integers(1, 256, size=n)
    ip_c = rng.integers(1, 256, size=n)

    # Trace/span IDs via numpy random bytes
    trace_raw = rng.bytes(n * 16)
    trace_ids = [trace_raw[i*16:(i+1)*16].hex() for i in range(n)]
    span_raw = rng.bytes(n * 8)
    span_ids = [span_raw[i*8:(i+1)*8].hex() for i in range(n)]

    events_count = rng.integers(0, 4, size=n, dtype=np.int32)
    links_count = np.zeros(n, dtype=np.int32)

    # Vectorized column construction
    svc_names_list = svc_names_arr.tolist()
    names_list = names.tolist()
    status_codes = np.where(is_error, "ERROR", "OK").tolist()
    status_msgs = np.where(is_error, "Error occurred", None).tolist()

    host_names_list = np.char.add(
        np.char.add(svc_names_arr.astype(str), "-"),
        np.char.add(host_nums.astype(str), ".example.com")
    ).tolist()
    host_ips = np.char.add(
        "10.0.",
        np.char.add(np.char.add(ip_b.astype(str), "."), ip_c.astype(str))
    ).tolist()

    http_method_vals = HTTP_METHODS[http_method_idx]
    http_route_vals = HTTP_ROUTES[http_route_idx]
    http_status_vals = HTTP_STATUS_CODES[http_status_idx]

    http_methods = np.where(is_server, http_method_vals, None).tolist()
    http_routes_list = np.where(is_server, http_route_vals, None).tolist()
    http_statuses = np.where(is_server, http_status_vals, None).tolist()
    http_urls = np.where(is_server, np.char.add("https://api.example.com", http_route_vals), None).tolist()

    attrs = np.char.add(np.char.add('{"service.name": "', svc_names_arr.astype(str)), '"}').tolist()
    res_attrs = ['{}'] * n

    db_systems = np.where(is_client, "postgresql", None).tolist()
    db_names_list = np.where(is_client, "production", None).tolist()
    db_ops = np.where(is_client, names, None).tolist()

    msg_systems = np.where(is_msg, "kafka", None).tolist()
    msg_dests = np.where(is_msg, "events", None).tolist()
    msg_ops = np.where(is_producer, "publish", np.where(is_consumer, "consume", None)).tolist()

    exc_types = np.where(is_error, "RuntimeError", None).tolist()
    exc_msgs = np.where(is_error, "Something went wrong", None).tolist()

    # Compute date partition column
    timestamps_s = span_starts // 1_000_000_000
    days_since_epoch = (timestamps_s // 86400).astype(np.int32)
    hours_of_day = ((timestamps_s % 86400) // 3600).astype(np.int8)
    unique_days = np.unique(days_since_epoch)
    day_to_str = {}
    for d in unique_days:
        dt = datetime.fromtimestamp(int(d) * 86400, tz=timezone.utc)
        day_to_str[int(d)] = dt.strftime("%Y-%m-%d")
    date_col = pa.array([day_to_str[int(d)] for d in days_since_epoch], type=pa.string())
    hour_col = pa.array(hours_of_day, type=pa.int8())

    table = pa.table({
        "trace_id": pa.array(trace_ids, type=pa.string()),
        "span_id": pa.array(span_ids, type=pa.string()),
        "parent_span_id": pa.nulls(n, type=pa.string()),
        "start_time_unix_nano": pa.array(span_starts),
        "end_time_unix_nano": pa.array(span_ends),
        "duration_ns": pa.array(durations),
        "name": pa.array(names_list, type=pa.string()),
        "kind": pa.array(kinds_arr.tolist()).dictionary_encode().cast(_span_kind_dict),
        "status_code": pa.array(status_codes).dictionary_encode().cast(_status_code_dict),
        "status_message": pa.array(status_msgs, type=pa.string()),
        "service_name": pa.array(svc_names_list, type=pa.string()),
        "service_namespace": pa.array(["production"] * n, type=pa.string()),
        "service_version": pa.array(["1.0.0"] * n, type=pa.string()),
        "host_name": pa.array(host_names_list, type=pa.string()),
        "host_ip": pa.array(host_ips, type=pa.string()),
        "attributes_json": pa.array(attrs, type=pa.string()),
        "resource_attributes_json": pa.array(res_attrs, type=pa.string()),
        "http_method": pa.array(http_methods, type=pa.string()),
        "http_status_code": pa.array(http_statuses, type=pa.int16()),
        "http_url": pa.array(http_urls, type=pa.string()),
        "http_route": pa.array(http_routes_list, type=pa.string()),
        "http_target": pa.nulls(n, type=pa.string()),
        "db_system": pa.array(db_systems, type=pa.string()),
        "db_name": pa.array(db_names_list, type=pa.string()),
        "db_operation": pa.array(db_ops, type=pa.string()),
        "db_statement": pa.nulls(n, type=pa.string()),
        "rpc_system": pa.nulls(n, type=pa.string()),
        "rpc_service": pa.nulls(n, type=pa.string()),
        "rpc_method": pa.nulls(n, type=pa.string()),
        "messaging_system": pa.array(msg_systems, type=pa.string()),
        "messaging_destination": pa.array(msg_dests, type=pa.string()),
        "messaging_operation": pa.array(msg_ops, type=pa.string()),
        "exception_type": pa.array(exc_types, type=pa.string()),
        "exception_message": pa.array(exc_msgs, type=pa.string()),
        "events_count": pa.array(events_count),
        "links_count": pa.array(links_count),
        "date": date_col,
        "hour": hour_col,
    })

    return table

print("generate_batch_vectorized() defined")
print("  - Exponential duration (median ~50ms, clipped 1ms-30s)")
print("  - 5% error rate")
print("  - Rejection sampling for service-operation matching")


## 2. Connect + write `date=` partitions (no hour=)


In [ ]:
s3_kwargs = CFG.pyarrow_s3_kwargs() if callable(getattr(CFG, "pyarrow_s3_kwargs", None)) else CFG.pyarrow_s3_kwargs
s3 = pafs.S3FileSystem(**s3_kwargs)
base_path = SPANS_ROOT
print(f"Connected  endpoint_set={bool(S3_ENDPOINT)}  base_path={base_path}")

def write_partitioned(table, root_path, batch_id, chunk_idx=None, max_retries=5):
    """{root}/date=YYYY-MM-DD/batch_NNNN.parquet — matches validation discovery."""
    sort_indices = pc.sort_indices(table, sort_keys=[("start_time_unix_nano", "ascending")])
    table = table.take(sort_indices)
    date_col = table.column("date")
    data_cols = [c for c in table.column_names if c not in ("date", "hour")]
    unique_dates = pc.unique(date_col).to_pylist()
    total_bytes = 0
    for date_val in unique_dates:
        mask = pc.equal(date_col, date_val)
        part = table.filter(mask).select(data_cols)
        date_str = date_val if isinstance(date_val, str) else str(date_val)
        chunk_tag = f"_chunk_{chunk_idx:02d}" if chunk_idx is not None else ""
        rel = f"{root_path}/date={date_str}/batch_{batch_id:04d}{chunk_tag}.parquet"
        for attempt in range(max_retries):
            try:
                with s3.open_output_stream(rel) as out:
                    pq.write_table(
                        part, out,
                        compression=COMPRESSION,
                        compression_level=COMPRESSION_LEVEL,
                        row_group_size=ROW_GROUP_SIZE,
                    )
                total_bytes += part.nbytes
                break
            except Exception:
                if attempt + 1 >= max_retries:
                    raise
                time.sleep(0.5 * (attempt + 1))
    return total_bytes


## 3. Generate + active-dataset marker


In [ ]:
end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(days=DURATION_DAYS)
time_range_ns = DURATION_DAYS * 24 * 3600 * 1_000_000_000
start_ns = int(start_time.timestamp() * 1_000_000_000)
num_batches = max(1, (TOTAL_SPANS + CHUNK_SIZE - 1) // CHUNK_SIZE)

print("=" * 60)
print("Generate → s3://%s/date=*/batch_*.parquet" % base_path)
print("=" * 60)
print(f"spans={TOTAL_SPANS:,}  batches={num_batches}  days={DURATION_DAYS}")
sys.stdout.flush()

overall_start = time.time()
total_bytes = 0
total_spans_written = 0
remaining = TOTAL_SPANS
dates_seen = set()

for batch_id in range(num_batches):
    chunk_size = min(CHUNK_SIZE, remaining)
    remaining -= chunk_size
    rng = np.random.default_rng(seed=batch_id * 10000)
    t0 = time.time()
    table = generate_batch_vectorized(chunk_size, rng, start_ns, time_range_ns)
    gen_time = time.time() - t0
    for d in pc.unique(table.column("date")).to_pylist():
        dates_seen.add(str(d))
    t1 = time.time()
    write_partitioned(table, base_path, batch_id, chunk_idx=0)
    write_time = time.time() - t1
    total_bytes += max(table.nbytes // 4, 1)
    total_spans_written += chunk_size
    del table
    print(f"  [{batch_id+1}/{num_batches}] {chunk_size:,}  gen={gen_time:.1f}s write={write_time:.1f}s")
    sys.stdout.flush()

marker = {
    "dataset": PREFIX,
    "updated_at": datetime.now(timezone.utc).isoformat(),
    "phase": "notebook",
    "total_spans": total_spans_written,
    "total_bytes": total_bytes,
}
marker_key = f"{BUCKET}/_active_dataset.json"
with s3.open_output_stream(marker_key) as f:
    f.write(json.dumps(marker).encode())

print("=" * 60)
print(f"Done in {time.time()-overall_start:.1f}s  spans={total_spans_written:,}")
print(f"Path:    s3://{base_path}/")
print(f"Dates:   {sorted(dates_seen)}")
print(f"Marker:  s3://{marker_key}  dataset={PREFIX!r}")
print()
print("→ Dask_S3_Validation next:")
print(f"   export SPANS_DATE={sorted(dates_seen)[-1] if dates_seen else 'YYYY-MM-DD'}")
print(f"   # or in notebook: SPANS_DATE = {sorted(dates_seen)[-1]!r}")
print(f"   list_span_parquet_keys(CFG)  # should list batch_*.parquet")


## 4. Smoke: same load path as validation (optional)


In [ ]:
from cluster_env import list_span_parquet_keys, load_active_spans_ddf
from dask.distributed import Client

keys = list_span_parquet_keys(CFG)
print(f"files={len(keys)} sample={keys[:3]}")
assert keys, "nothing listed — endpoint/creds/path mismatch with write"

day = sorted(dates_seen)[-1] if dates_seen else None
ddf = load_active_spans_ddf(CFG, date=day, aggregate_files=True)
print("partitions", ddf.npartitions, "cols", list(ddf.columns)[:8])
client = Client(CFG.dask_scheduler)
n = int(ddf.shape[0].compute())  # not map_partitions(len).sum() — bare int has no .sum()
print(f"rows≈{n:,} workers={len(client.scheduler_info()['workers'])}")
print("✔ Generator aligned with Dask_S3_Validation")
